In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
from google.colab import files
uploaded = files.upload()

Saving y_test.csv to y_test.csv
Saving X_test.csv to X_test.csv
Saving X_train.csv to X_train.csv
Saving X_val.csv to X_val.csv
Saving y_train.csv to y_train.csv
Saving y_val.csv to y_val.csv


In [3]:
import pandas as pd

X_train = pd.read_csv("/content/X_train.csv")
y_train = pd.read_csv("/content/y_train.csv").squeeze()

X_val = pd.read_csv("/content/X_val.csv")
y_val = pd.read_csv("/content/y_val.csv").squeeze()

X_test = pd.read_csv("/content/X_test.csv")
y_test = pd.read_csv("/content/y_test.csv").squeeze()

print("All datasets loaded successfully!")
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

All datasets loaded successfully!
(23714, 49) (23714,)
(5082, 49) (5082,)
(5082, 49) (5082,)


In [4]:
def evaluate_model(y_true, y_pred, name="Model"):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"{name} Performance:")
    print(f"  MAE : {mae:.2f}")
    print(f"  RMSE: {rmse:.2f}")
    print(f"  R²  : {r2:.3f}")
    print("-" * 30)

In [5]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

X_train["station"] = le.fit_transform(X_train["station"].astype(str))
X_val["station"] = le.transform(X_val["station"].astype(str))
X_test["station"] = le.transform(X_test["station"].astype(str))

print("Station column converted successfully!")

Station column converted successfully!


In [6]:
# Train the random forest regressor
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_val_pred_rf = rf.predict(X_val)

evaluate_model(y_val, y_val_pred_rf, "Random Forest (Validation)")

Random Forest (Validation) Performance:
  MAE : 3.73
  RMSE: 10.04
  R²  : 0.979
------------------------------


In [7]:
# Test evaluation
y_test_pred_rf = rf.predict(X_test)
evaluate_model(y_test, y_test_pred_rf, "Random Forest (Test)")

Random Forest (Test) Performance:
  MAE : 3.72
  RMSE: 7.69
  R²  : 0.993
------------------------------


In [8]:
# Train gradient boosting
gbr = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

gbr.fit(X_train, y_train)

y_val_pred_gbr = gbr.predict(X_val)

evaluate_model(y_val, y_val_pred_gbr, "Gradient Boosting (Validation)")

Gradient Boosting (Validation) Performance:
  MAE : 4.89
  RMSE: 10.10
  R²  : 0.979
------------------------------


In [9]:
y_test_pred_gbr = gbr.predict(X_test)
evaluate_model(y_test, y_test_pred_gbr, "Gradient Boosting (Test)")

Gradient Boosting (Test) Performance:
  MAE : 4.96
  RMSE: 8.78
  R²  : 0.991
------------------------------


In [10]:
# View results in a nice table
results_tree = pd.DataFrame({
    "Model": [
        "Random Forest (Validation)",
        "Random Forest (Test)",
        "Gradient Boosting (Validation)",
        "Gradient Boosting (Test)"
    ],
    "MAE": [
        mean_absolute_error(y_val, y_val_pred_rf),
        mean_absolute_error(y_test, y_test_pred_rf),
        mean_absolute_error(y_val, y_val_pred_gbr),
        mean_absolute_error(y_test, y_test_pred_gbr)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_val, y_val_pred_rf)),
        np.sqrt(mean_squared_error(y_test, y_test_pred_rf)),
        np.sqrt(mean_squared_error(y_val, y_val_pred_gbr)),
        np.sqrt(mean_squared_error(y_test, y_test_pred_gbr))
    ],
    "R2": [
        r2_score(y_val, y_val_pred_rf),
        r2_score(y_test, y_test_pred_rf),
        r2_score(y_val, y_val_pred_gbr),
        r2_score(y_test, y_test_pred_gbr)
    ]
})

results_tree.round(3)

,Model,MAE,RMSE,R2
0,Random Forest (Validation),3.732,10.040,0.979
1,Random Forest (Test),3.716,7.693,0.993
2,Gradient Boosting (Validation),4.891,10.098,0.979
3,Gradient Boosting (Test),4.960,8.784,0.991


In [11]:
# Save models
import joblib
import os

# Create model directory
os.makedirs("../models", exist_ok=True)

# Save Random Forest model
joblib.dump(rf, "../models/random_forest_pm25.pkl")

# Save Gradient Boosting model (XGBoost-style)
joblib.dump(gbr, "../models/gradient_boosting_pm25.pkl")

print("Models saved successfully.")

Models saved successfully.


In [13]:
import joblib
from pathlib import Path

Path("/content/models").mkdir(parents=True, exist_ok=True)

joblib.dump(rf, "/content/models/random_forest_pm25.pkl")

print("Random Forest model saved successfully!")

Random Forest model saved successfully!


In [14]:
from google.colab import files

files.download("/content/models/random_forest_pm25.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>